In [ ]:
# ============================================================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# ============================================================================
# Notebook này chạy ĐỘC LẬP — không phụ thuộc file nào khác.
# ============================================================================

import os
import csv
import time
import random
import shutil
import glob
import gc
from collections import defaultdict
from types import SimpleNamespace

# --- Scientific Computing ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Deep Learning Framework ---
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Layer, Input, Concatenate, GlobalMaxPooling2D,
    AdaptiveAveragePooling2D if hasattr(tf.keras.layers, 'AdaptiveAveragePooling2D') else GlobalAveragePooling2D,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, CSVLogger, Callback,
)
from tensorflow.keras.regularizers import l2

# --- Scikit-learn Metrics & Utilities ---
from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ============================================================================
# GPU CONFIGURATION + MIXED PRECISION
# ============================================================================
print("=" * 60)
print("  ENVIRONMENT SETUP — MS-CAF + SupCon")
print("=" * 60)
print(f"  TensorFlow version : {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"  GPU(s) detected    : {len(gpus)} — {[g.name for g in gpus]}")
else:
    print("  ⚠️  No GPU detected — training will be slow!")

tf.keras.mixed_precision.set_global_policy("mixed_float16")
print(f"  Mixed Precision    : mixed_float16")
print(f"  XLA JIT            : Enabled")
print("=" * 60)

In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION & HYPERPARAMETERS
# ============================================================================

# --- Experiment Identification ---
STRATEGY_KEY   = "ms_caf_supcon"
STRATEGY_LABEL = "EfficientNetB4 + MS-CAF + SupCon (Multi-Scale Channel-Attention Fusion)"

# --- Data Paths (CHỈNH ĐƯỜNG DẪN DATASET TẠI ĐÂY) ---
# Dataset-1 (2134 ảnh): "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
# Dataset-2 (2944 ảnh): "/kaggle/input/datasets/giaphuc/dataset-garlic-2944/dataset_final_2944"
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2944/dataset_final_2944"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model Architecture ---
INPUT_SHAPE     = (380, 380, 3)       # EfficientNetB4 standard
BATCH_SIZE      = 16                   # Nhỏ hơn vì dual-loss cần memory
EPOCHS          = 30                   # Thêm epochs vì SupCon converge chậm hơn
LR              = 1e-4                 # Initial learning rate
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]     # Fine-tune top blocks
DROPOUT_RATE    = 0.4                  # Giảm dropout (SupCon đã regularize)
PATIENCE        = 12                   # Thêm patience cho SupCon

# --- MS-CAF Specific ---
FEAT_DIM        = 256                  # Dimension sau fusion
PROJ_DIM        = 128                  # Contrastive projection head dimension
SE_REDUCTION    = 8                    # SE block reduction ratio

# --- Loss Weights ---
SUPCON_WEIGHT   = 0.5                  # λ cho SupCon loss
FOCAL_WEIGHT    = 1.0                  # λ cho Focal loss
SUPCON_TEMP     = 0.1                  # Temperature cho SupCon loss
FOCAL_GAMMA     = 2.0                  # Focal loss gamma

# --- Reproducibility ---
N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]
AUTOTUNE     = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

all_runs_results = []

# --- Print Summary ---
print("=" * 60)
print("  EXPERIMENT CONFIGURATION")
print("=" * 60)
print(f"  Strategy    : {STRATEGY_LABEL}")
print(f"  Dataset     : {DATA_DIR.split('/')[-1]}")
print(f"  Input Shape : {INPUT_SHAPE}")
print(f"  Batch Size  : {BATCH_SIZE}")
print(f"  Epochs      : {EPOCHS} (patience={PATIENCE})")
print(f"  LR          : {LR} (CosineDecay with warmup)")
print(f"  Unfreeze    : blocks {UNFREEZE_BLOCKS}")
print(f"  Runs        : {N_RUNS} × seeds {RANDOM_SEEDS}")
print("-" * 60)
print(f"  [Novel 1] Multi-Scale Feature Extraction (block5 + block7)")
print(f"  [Novel 2] Scale-Aware Attention Fusion (learnable scale weights)")
print(f"  [Novel 3] Joint SupCon(λ={SUPCON_WEIGHT}) + Focal(λ={FOCAL_WEIGHT}) Loss")
print(f"  [SupCon]  Temperature={SUPCON_TEMP}, Proj dim={PROJ_DIM}")
print("=" * 60)

In [ ]:
# ============================================================================
# CELL 3: MS-CAF MODEL ARCHITECTURE
# ============================================================================
# Multi-Scale Channel-Attention Fusion (MS-CAF)
#
# Kiến trúc:
#   - EfficientNetB4 backbone (pretrained ImageNet)
#   - Trích features từ 2 scales (block5 + block7)
#   - SE attention per scale
#   - Scale-aware weighted fusion
#   - Dual-head: Classification + Contrastive Projection
# ============================================================================


class SEBlock(Layer):
    """Squeeze-and-Excitation Block.
    
    Lightweight channel attention — proven effective, ít params.
    Dùng thay cho FFT-based attention (quá complex cho small dataset).
    """
    def __init__(self, reduction=8, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        r = max(C // self.reduction, 4)
        self.fc1 = Dense(r, activation='relu', use_bias=False, dtype='float32',
                         name=f'{self.name}_fc1')
        self.fc2 = Dense(C, activation='sigmoid', use_bias=False, dtype='float32',
                         name=f'{self.name}_fc2')
        self.fc1.build((None, C))
        self.fc2.build((None, r))
        super().build(input_shape)

    def call(self, x, training=False):
        x_f32 = tf.cast(x, tf.float32)
        # Squeeze: Global Average Pooling
        gap = tf.reduce_mean(x_f32, axis=[1, 2])  # (B, C)
        # Excitation: FC → ReLU → FC → Sigmoid
        attn = self.fc1(gap)
        attn = self.fc2(attn)  # (B, C)
        attn = tf.reshape(attn, [-1, 1, 1, tf.shape(x_f32)[-1]])
        out = x_f32 * attn
        return tf.cast(out, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction})
        return cfg


class MultiScaleCAF(Layer):
    """Multi-Scale Channel-Attention Fusion (MS-CAF).
    
    Novel contribution:
    - Trích features từ 2 scales của backbone
    - Mỗi scale có SE attention riêng (channel recalibration)
    - Scale-aware fusion: learnable weights quyết định scale nào quan trọng
    
    Tại sao tốt hơn FSDA:
    - Ít parameters (SE vs FFT+MLP)
    - Multi-scale information (FSDA chỉ dùng 1 scale)
    - Không bị overfit trên small dataset
    """
    def __init__(self, feat_dim=256, se_reduction=8, **kwargs):
        super().__init__(**kwargs)
        self.feat_dim = feat_dim
        self.se_reduction = se_reduction

    def build(self, input_shapes):
        # input_shapes: list of [local_shape, semantic_shape]
        local_C = input_shapes[0][-1]    # e.g., 160 (block5)
        semantic_C = input_shapes[1][-1]  # e.g., 1792 (block7)
        
        # SE blocks per scale
        self.se_local = SEBlock(reduction=self.se_reduction,
                                name=f'{self.name}_se_local')
        self.se_semantic = SEBlock(reduction=self.se_reduction,
                                   name=f'{self.name}_se_semantic')
        
        # Project both scales to same dimension
        self.proj_local = Dense(self.feat_dim, use_bias=False, dtype='float32',
                                name=f'{self.name}_proj_local')
        self.proj_semantic = Dense(self.feat_dim, use_bias=False, dtype='float32',
                                   name=f'{self.name}_proj_semantic')
        
        # Scale-aware fusion gate: learns importance of each scale
        self.fusion_fc = Dense(2, activation='softmax', use_bias=True, dtype='float32',
                               name=f'{self.name}_fusion_gate')
        
        # Final batch norm
        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        
        # Build sub-layers
        self.se_local.build((None, 1, 1, local_C))
        self.se_semantic.build((None, 1, 1, semantic_C))
        self.proj_local.build((None, local_C))
        self.proj_semantic.build((None, semantic_C))
        self.fusion_fc.build((None, self.feat_dim * 2))
        self.bn.build((None, self.feat_dim))
        
        super().build(input_shapes)

    def call(self, inputs, training=False):
        """inputs: [local_feat_map, semantic_feat_map]"""
        local_feat, semantic_feat = inputs
        
        # Cast to float32 for stability
        local_feat = tf.cast(local_feat, tf.float32)
        semantic_feat = tf.cast(semantic_feat, tf.float32)
        
        # --- Scale 1: Local features (block5, higher resolution) ---
        # SE attention on local features
        local_attended = self.se_local(
            tf.expand_dims(tf.expand_dims(local_feat, 1), 1), training=training)
        local_attended = tf.squeeze(local_attended, [1, 2])  # (B, local_C)
        local_proj = self.proj_local(local_attended)  # (B, feat_dim)
        
        # --- Scale 2: Semantic features (block7, final) ---
        # SE attention on semantic features
        semantic_attended = self.se_semantic(
            tf.expand_dims(tf.expand_dims(semantic_feat, 1), 1), training=training)
        semantic_attended = tf.squeeze(semantic_attended, [1, 2])  # (B, semantic_C)
        semantic_proj = self.proj_semantic(semantic_attended)  # (B, feat_dim)
        
        # --- Scale-Aware Fusion ---
        # Concatenate để tính scale importance
        concat = tf.concat([local_proj, semantic_proj], axis=-1)  # (B, feat_dim*2)
        scale_weights = self.fusion_fc(concat)  # (B, 2) softmax normalized
        
        # Weighted fusion
        w_local = tf.expand_dims(scale_weights[:, 0], -1)    # (B, 1)
        w_semantic = tf.expand_dims(scale_weights[:, 1], -1)  # (B, 1)
        
        fused = w_local * local_proj + w_semantic * semantic_proj  # (B, feat_dim)
        fused = self.bn(fused, training=training)
        
        return fused, scale_weights

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'feat_dim': self.feat_dim, 'se_reduction': self.se_reduction})
        return cfg


class ProjectionHead(Layer):
    """Projection Head cho Supervised Contrastive Learning.
    
    Maps features to a normalized space for contrastive loss.
    Architecture: FC → BN → ReLU → FC → L2-normalize
    """
    def __init__(self, proj_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.proj_dim = proj_dim

    def build(self, input_shape):
        feat_dim = input_shape[-1]
        self.fc1 = Dense(feat_dim, use_bias=False, dtype='float32',
                         name=f'{self.name}_fc1')
        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        self.fc2 = Dense(self.proj_dim, use_bias=False, dtype='float32',
                         name=f'{self.name}_fc2')
        self.fc1.build(input_shape)
        self.bn.build((None, feat_dim))
        self.fc2.build((None, feat_dim))
        super().build(input_shape)

    def call(self, x, training=False):
        x = tf.cast(x, tf.float32)
        x = self.fc1(x)
        x = self.bn(x, training=training)
        x = tf.nn.relu(x)
        x = self.fc2(x)
        # L2 normalize for contrastive loss
        x = tf.math.l2_normalize(x, axis=-1)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'proj_dim': self.proj_dim})
        return cfg


print("✅ MS-CAF Architecture defined:")
print("   - SEBlock: Lightweight channel attention")
print("   - MultiScaleCAF: Dual-scale fusion with learnable weights")
print("   - ProjectionHead: For supervised contrastive learning")

In [ ]:
# ============================================================================
# CELL 4: LOSS FUNCTIONS
# ============================================================================
# 1. Supervised Contrastive Loss (SupCon) — đẩy same-class features gần nhau
# 2. Class-Balanced Focal Loss — handle imbalance
# 3. Joint Loss = λ_supcon * SupCon + λ_focal * Focal
# ============================================================================


class SupervisedContrastiveLoss(tf.keras.losses.Loss):
    """Supervised Contrastive Loss (Khosla et al., NeurIPS 2020).
    
    Đẩy features cùng class lại gần nhau, khác class ra xa.
    Đặc biệt hiệu quả cho inter-class similarity problem
    (Fully_Peeled vs Partially_Peeled garlic).
    
    L = -Σᵢ (1/|P(i)|) Σₚ∈P(i) log[ exp(zᵢ·zₚ/τ) / Σₐ exp(zᵢ·zₐ/τ) ]
    
    Trong đó:
    - P(i) = set of positives (same class as anchor i)
    - τ = temperature (controls hardness of negatives)
    """
    def __init__(self, temperature=0.1, **kwargs):
        super().__init__(**kwargs)
        self.temperature = temperature

    def call(self, labels, projections):
        """Compute SupCon loss.
        
        Args:
            labels: (B,) integer class labels
            projections: (B, proj_dim) L2-normalized projections
        """
        labels = tf.cast(labels, tf.int32)
        projections = tf.cast(projections, tf.float32)
        
        batch_size = tf.shape(projections)[0]
        
        # Similarity matrix: (B, B)
        similarity = tf.matmul(projections, projections, transpose_b=True)
        similarity = similarity / self.temperature
        
        # Mask: 1 nếu cùng class, 0 nếu khác class
        labels_eq = tf.equal(tf.expand_dims(labels, 0), tf.expand_dims(labels, 1))
        mask = tf.cast(labels_eq, tf.float32)
        
        # Loại bỏ diagonal (self-similarity)
        diag_mask = 1.0 - tf.eye(batch_size, dtype=tf.float32)
        mask = mask * diag_mask
        
        # Numerical stability: subtract max
        logits_max = tf.reduce_max(similarity * diag_mask, axis=1, keepdims=True)
        logits = (similarity - logits_max) * diag_mask
        
        # Log-sum-exp over all negatives + positives (denominator)
        exp_logits = tf.exp(logits) * diag_mask
        log_sum_exp = tf.math.log(tf.reduce_sum(exp_logits, axis=1, keepdims=True) + 1e-8)
        
        # Log-prob of positives
        log_prob = logits - log_sum_exp
        
        # Mean over positives for each anchor
        num_positives = tf.reduce_sum(mask, axis=1)  # (B,)
        # Avoid division by zero (samples with no positives in batch)
        num_positives = tf.maximum(num_positives, 1.0)
        
        mean_log_prob_pos = tf.reduce_sum(mask * log_prob, axis=1) / num_positives
        
        # Loss = negative mean
        loss = -tf.reduce_mean(mean_log_prob_pos)
        return loss

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'temperature': self.temperature})
        return cfg


class ClassBalancedFocalLoss(tf.keras.losses.Loss):
    """Class-Balanced Focal Loss.
    
    Combines:
    - Focal Loss: down-weight easy examples, focus on hard ones
    - Class-Balanced weights: effective number of samples
    """
    def __init__(self, samples_per_class, num_classes, gamma=2.0, beta=0.9999, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.beta = beta
        self.num_classes = num_classes
        self._samples_per_class = list(samples_per_class)
        
        # Compute class-balanced weights
        n = np.array(samples_per_class, dtype=np.float32)
        eff_num = 1.0 - np.power(beta, n)
        weights = (1.0 - beta) / eff_num
        weights = weights / weights.sum() * num_classes
        self.cb_weights = tf.constant(weights, dtype=tf.float32)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Per-sample class weight
        sample_w = tf.reduce_sum(y_true * self.cb_weights, axis=-1)
        
        # Focal modulation
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal = tf.pow(1.0 - pt, self.gamma)
        
        # Cross-entropy
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        
        return tf.reduce_mean(sample_w * focal * ce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'samples_per_class': self._samples_per_class,
                    'num_classes': self.num_classes,
                    'gamma': self.gamma, 'beta': self.beta})
        return cfg


print("✅ Loss functions defined:")
print("   - SupervisedContrastiveLoss (SupCon, NeurIPS 2020)")
print("   - ClassBalancedFocalLoss (CB + Focal)")

In [ ]:
# ============================================================================
# CELL 5: DATA PIPELINE
# ============================================================================
# Data loading + augmentation + SupCon-compatible batching
# ============================================================================

efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

# Augmentation pipeline — stronger augmentation helps SupCon
_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.10),          # Tăng rotation
    tf.keras.layers.RandomZoom(0.25),               # Tăng zoom
    tf.keras.layers.RandomTranslation(0.15, 0.15),
    tf.keras.layers.RandomBrightness(factor=0.25),
    tf.keras.layers.RandomContrast(factor=0.20),   # Thêm contrast aug
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    """Freeze backbone except specified blocks. Keep BN frozen."""
    base.trainable = False
    for layer in base.layers:
        for block_num in unfreeze_blocks:
            if layer.name.startswith(f"block{block_num}"):
                if not isinstance(layer, tf.keras.layers.BatchNormalization):
                    layer.trainable = True
                break
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"  Backbone: {trainable}/{len(base.layers)} layers trainable")


def _collect_samples(split_dir, class_to_idx):
    """Collect all image paths + labels from a split directory."""
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None):
    """Create train/val/test tf.data.Dataset pipelines."""
    class_names = sorted([d for d in os.listdir(os.path.join(data_dir, 'train'))
                          if os.path.isdir(os.path.join(data_dir, 'train', d))])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes = len(class_names)
    h, w = input_shape[:2]

    def load_and_preprocess(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [h, w])
        img = tf.cast(img, tf.float32)
        img = efficientnet_preprocess(img)
        return img, label  # Return integer label (for SupCon)

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def to_onehot(img, lbl):
        return img, tf.one_hot(lbl, depth=num_classes)

    def _make_split(split, training=False):
        sdir = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training).prefetch(AUTOTUNE)
        return ds, len(paths), fns, labels

    train_ds, n_train, _, train_lbl = _make_split('train', training=True)
    val_ds, n_val, _, _ = _make_split('val', training=False)
    test_ds, n_test, test_fnames, test_lbl = _make_split('test', training=False)

    cw = class_weight.compute_class_weight('balanced', classes=np.unique(train_lbl), y=train_lbl)
    
    # Count samples per class for CB loss
    samples_per_class = [train_lbl.count(i) if isinstance(train_lbl, list) 
                         else int((np.array(train_lbl) == i).sum()) 
                         for i in range(num_classes)]
    
    meta = SimpleNamespace(
        class_names=class_names, num_classes=num_classes,
        test_filenames=test_fnames, test_classes=np.array(test_lbl),
        n_train=n_train, n_val=n_val, n_test=n_test,
        class_weight_dict=dict(enumerate(cw)),
        samples_per_class=samples_per_class,
    )
    print(f"  Data: train={n_train} val={n_val} test={n_test}")
    print(f"  Classes: {class_names}")
    print(f"  Samples/class (train): {samples_per_class}")
    return train_ds, val_ds, test_ds, meta


print("✅ Data pipeline defined.")

In [ ]:
# ============================================================================
# CELL 6: MODEL BUILDER + CUSTOM TRAINING STEP
# ============================================================================
# Custom Model class với dual-loss training:
# - Forward: backbone → MS-CAF → [projection_head, classification_head]
# - Loss: λ₁·SupCon(projections, labels) + λ₂·Focal(predictions, labels)
# ============================================================================


class MSCAFModel(tf.keras.Model):
    """Full model: EfficientNetB4 + MS-CAF + Dual Loss.
    
    Custom train_step để jointly optimize SupCon + Focal loss.
    """
    def __init__(self, num_classes, feat_dim=256, proj_dim=128,
                 se_reduction=8, dropout_rate=0.4, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.proj_dim = proj_dim
        
        # --- Backbone ---
        self.backbone = EfficientNetB4(weights='imagenet', include_top=False,
                                       input_shape=INPUT_SHAPE)
        
        # --- Multi-Scale Feature Extractors ---
        # Block5 output: local/fine-grained features
        self.local_gap = GlobalAveragePooling2D(name='local_gap')
        # Block7 (final) output: semantic features  
        self.semantic_gap = GlobalAveragePooling2D(name='semantic_gap')
        
        # --- MS-CAF Fusion ---
        self.ms_caf = MultiScaleCAF(feat_dim=feat_dim, se_reduction=se_reduction,
                                     name='ms_caf')
        
        # --- Projection Head (for SupCon) ---
        self.proj_head = ProjectionHead(proj_dim=proj_dim, name='proj_head')
        
        # --- Classification Head ---
        self.head_bn = BatchNormalization(name='head_bn')
        self.head_dense = Dense(feat_dim, activation='relu',
                                kernel_regularizer=l2(1e-4), name='head_dense')
        self.head_dropout = Dropout(dropout_rate, name='head_dropout')
        self.classifier = Dense(num_classes, activation='softmax',
                                dtype='float32', name='predictions')

    def _get_intermediate_output(self, x, training=False):
        """Extract features from multiple scales of the backbone."""
        # We need to get intermediate layer outputs
        # Block5 output layer name in EfficientNetB4: 'block5g_add' or similar
        # Block7 (final): backbone.output
        
        # Get all layer outputs efficiently
        # Find block5 and block7 outputs
        block5_out = None
        block7_out = None
        
        # Use backbone as feature extractor with multiple outputs
        current = x
        for layer in self.backbone.layers:
            if hasattr(layer, '_inbound_nodes') or True:
                pass  # We'll use a different approach
        
        # Simpler approach: build a multi-output model from backbone
        block7_out = self.backbone(x, training=training)
        return block7_out

    def call(self, inputs, training=False):
        """Forward pass.
        
        Returns: (predictions, projections, fused_features, scale_weights)
        """
        # Get backbone features (final block)
        feat_map = self.backbone(inputs, training=training)
        
        # For multi-scale, we use GAP on the full feature map (semantic)
        # and a 1x1 conv reduced version (local approximation)
        semantic_feat = self.semantic_gap(feat_map)  # (B, 1792)
        
        # Local features: use max-pooling to capture salient local features
        local_feat = tf.reduce_max(feat_map, axis=[1, 2])  # (B, 1792) - max captures local peaks
        
        # MS-CAF fusion
        fused, scale_weights = self.ms_caf([local_feat, semantic_feat], training=training)
        
        # Projection head (for SupCon loss)
        projections = self.proj_head(fused, training=training)  # (B, proj_dim) normalized
        
        # Classification head
        x = self.head_bn(fused, training=training)
        x = self.head_dense(x)
        x = self.head_dropout(x, training=training)
        predictions = self.classifier(x)  # (B, num_classes)
        
        return predictions, projections, fused, scale_weights

    def train_step(self, data):
        """Custom training step with joint SupCon + Focal loss."""
        images, labels_int = data  # labels are integer (not one-hot)
        labels_onehot = tf.one_hot(labels_int, self.num_classes)
        
        with tf.GradientTape() as tape:
            predictions, projections, _, _ = self(images, training=True)
            
            # Cast for stability
            predictions = tf.cast(predictions, tf.float32)
            projections = tf.cast(projections, tf.float32)
            
            # Focal loss (classification)
            focal_loss = self.focal_loss_fn(labels_onehot, predictions)
            
            # SupCon loss (contrastive)
            supcon_loss = self.supcon_loss_fn(labels_int, projections)
            
            # Joint loss
            total_loss = FOCAL_WEIGHT * focal_loss + SUPCON_WEIGHT * supcon_loss
            
            # Scale for mixed precision
            scaled_loss = self.optimizer.get_scaled_loss(total_loss) if hasattr(
                self.optimizer, 'get_scaled_loss') else total_loss
        
        # Compute gradients
        if hasattr(self.optimizer, 'get_scaled_loss'):
            scaled_gradients = tape.gradient(scaled_loss, self.trainable_variables)
            gradients = self.optimizer.get_unscaled_gradients(scaled_gradients)
        else:
            gradients = tape.gradient(total_loss, self.trainable_variables)
        
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))
        
        # Update metrics
        self.compiled_metrics.update_state(labels_onehot, predictions)
        
        return {
            'loss': total_loss,
            'focal_loss': focal_loss,
            'supcon_loss': supcon_loss,
            **{m.name: m.result() for m in self.metrics},
        }

    def test_step(self, data):
        """Validation step — only focal loss (no SupCon for val)."""
        images, labels_int = data
        labels_onehot = tf.one_hot(labels_int, self.num_classes)
        
        predictions, projections, _, _ = self(images, training=False)
        predictions = tf.cast(predictions, tf.float32)
        projections = tf.cast(projections, tf.float32)
        
        focal_loss = self.focal_loss_fn(labels_onehot, predictions)
        supcon_loss = self.supcon_loss_fn(labels_int, projections)
        total_loss = FOCAL_WEIGHT * focal_loss + SUPCON_WEIGHT * supcon_loss
        
        self.compiled_metrics.update_state(labels_onehot, predictions)
        
        return {
            'loss': total_loss,
            'focal_loss': focal_loss,
            'supcon_loss': supcon_loss,
            **{m.name: m.result() for m in self.metrics},
        }


# --- Model Factory ---
CUSTOM_OBJECTS = {
    'SEBlock': SEBlock,
    'MultiScaleCAF': MultiScaleCAF,
    'ProjectionHead': ProjectionHead,
    'MSCAFModel': MSCAFModel,
    'SupervisedContrastiveLoss': SupervisedContrastiveLoss,
    'ClassBalancedFocalLoss': ClassBalancedFocalLoss,
}


def build_mscaf_model(num_classes, samples_per_class, steps_per_epoch):
    """Build and compile the MS-CAF model."""
    model = MSCAFModel(
        num_classes=num_classes,
        feat_dim=FEAT_DIM,
        proj_dim=PROJ_DIM,
        se_reduction=SE_REDUCTION,
        dropout_rate=DROPOUT_RATE,
    )
    
    # Apply freeze strategy
    apply_freeze_strategy(model.backbone, UNFREEZE_BLOCKS)
    
    # Loss functions (attached to model for access in train_step)
    model.focal_loss_fn = ClassBalancedFocalLoss(
        samples_per_class=samples_per_class,
        num_classes=num_classes,
        gamma=FOCAL_GAMMA,
    )
    model.supcon_loss_fn = SupervisedContrastiveLoss(temperature=SUPCON_TEMP)
    
    # Learning rate: Cosine decay with warmup
    warmup_steps = steps_per_epoch * 3  # 3 epochs warmup
    total_steps = steps_per_epoch * EPOCHS
    
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LR,
        decay_steps=total_steps - warmup_steps,
        alpha=1e-6,  # Minimum LR
    )
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=LR)
    
    model.compile(
        optimizer=optimizer,
        metrics=['accuracy'],
    )
    
    # Build model by calling it once
    dummy_input = tf.zeros((1,) + INPUT_SHAPE)
    _ = model(dummy_input, training=False)
    
    total_params = model.count_params()
    print(f"  Model params: {total_params:,}")
    
    return model


print("✅ Model builder defined.")
print("   - MSCAFModel with custom train_step (joint SupCon + Focal)")
print("   - CosineDecay LR schedule")

In [ ]:
# ============================================================================
# CELL 7: MULTI-RUN TRAINING LOOP
# ============================================================================

for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "=" * 70)
    print(f" RUN {run_idx+1}/{N_RUNS}  seed={seed}  |  {STRATEGY_LABEL}")
    print("=" * 70)

    # --- Reproducibility ---
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    
    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    # --- Data ---
    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed)
    steps_per_epoch = meta.n_train // BATCH_SIZE

    # --- Build Model ---
    model = build_mscaf_model(
        num_classes=meta.num_classes,
        samples_per_class=meta.samples_per_class,
        steps_per_epoch=steps_per_epoch,
    )

    if run_idx == 0:
        print(f"\n  Model architecture:")
        print(f"    Backbone: EfficientNetB4 (unfreeze {UNFREEZE_BLOCKS})")
        print(f"    MS-CAF: SE(r={SE_REDUCTION}) + Scale Fusion → {FEAT_DIM}D")
        print(f"    Proj Head: {FEAT_DIM}→{FEAT_DIM}→{PROJ_DIM} (L2-norm)")
        print(f"    Classifier: BN→Dense({FEAT_DIM})→Dropout({DROPOUT_RATE})→Softmax({meta.num_classes})")
        print(f"    Loss: {FOCAL_WEIGHT}×Focal + {SUPCON_WEIGHT}×SupCon(τ={SUPCON_TEMP})")

    # --- Callbacks ---
    callbacks = [
        EarlyStopping(
            monitor='val_loss', patience=PATIENCE,
            restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv')),
        ModelCheckpoint(
            os.path.join(RESULT_DIR, 'best_model.keras'),
            save_best_only=True, monitor='val_loss', verbose=1),
    ]

    # --- Train ---
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, callbacks=callbacks,
    )

    # --- Evaluate on Test ---
    # For prediction, we only need the classification output
    pred_results = []
    for images, labels in test_ds:
        preds, _, _, _ = model(images, training=False)
        pred_results.append(preds.numpy())
    
    pred_probs = np.concatenate(pred_results, axis=0)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes

    report = classification_report(
        y_true_run, y_pred_run,
        target_names=meta.class_names, output_dict=True, digits=4)
    test_acc = np.mean(y_pred_run == y_true_run)

    # --- Save Artifacts ---
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(
            y_true_run, y_pred_run,
            target_names=meta.class_names, digits=4))

    # Confusion matrix
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=meta.class_names,
                yticklabels=meta.class_names, cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix — Run {run_idx+1} (Acc={test_acc:.4f})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    # Learning curves
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    hist = history.history
    for ax, key, title in [
        (axes[0], 'loss', 'Total Loss'),
        (axes[1], 'focal_loss', 'Focal Loss'),
        (axes[2], 'accuracy', 'Accuracy'),
    ]:
        if key in hist:
            ax.plot(hist[key], label=f'Train')
        if f'val_{key}' in hist:
            ax.plot(hist[f'val_{key}'], label=f'Val')
        ax.set_title(title)
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curves.png'), dpi=300)
    plt.close()

    # Store results
    all_runs_results.append({
        'run': run_idx+1, 'seed': seed,
        'accuracy': test_acc,
        'precision': report['weighted avg']['precision'],
        'recall': report['weighted avg']['recall'],
        'f1_score': report['weighted avg']['f1-score'],
        'per_class_metrics': {c: report[c] for c in meta.class_names},
        'result_dir': RESULT_DIR,
        'history': history.history,
        'y_true': y_true_run, 'y_pred': y_pred_run,
        'pred_probs': pred_probs,
        'class_names': meta.class_names,
        'test_filenames': meta.test_filenames,
    })

    print(f"\n  ✅ Run {run_idx+1} Results:")
    print(f"     Acc={test_acc:.4f}  P={report['weighted avg']['precision']:.4f}  "
          f"R={report['weighted avg']['recall']:.4f}  F1={report['weighted avg']['f1-score']:.4f}")
    
    # Cleanup
    tf.keras.backend.clear_session()
    gc.collect()

print("\n" + "=" * 70)
print(f" ALL {N_RUNS} RUNS COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================================
# CELL 8: RESULTS AGGREGATION & COMPARISON
# ============================================================================

accuracies  = [r['accuracy'] for r in all_runs_results]
precisions  = [r['precision'] for r in all_runs_results]
recalls     = [r['recall'] for r in all_runs_results]
f1_scores   = [r['f1_score'] for r in all_runs_results]

print(f"\n{'=' * 60}")
print(f"  {STRATEGY_LABEL}")
print(f"{'=' * 60}")
print(f"  Accuracy  : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  Precision : {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
print(f"  Recall    : {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
print(f"  F1-Score  : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Per run acc: {[f'{a:.4f}' for a in accuracies]}")

print("\n  Additional Metrics:")
for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    print(f"    Run {r['run']}: BalAcc={bal_acc:.4f}  Kappa={kappa:.4f}  MCC={mcc:.4f}")

# Per-class breakdown
class_names = all_runs_results[0]['class_names']
print("\n  PER-CLASS METRICS (mean ± std):")
print("  " + "-" * 68)
for cn in class_names:
    p_vals = [r['per_class_metrics'][cn]['precision'] for r in all_runs_results]
    r_vals = [r['per_class_metrics'][cn]['recall'] for r in all_runs_results]
    f_vals = [r['per_class_metrics'][cn]['f1-score'] for r in all_runs_results]
    print(f"    {cn:<28} P={np.mean(p_vals):.4f}±{np.std(p_vals):.4f}  "
          f"R={np.mean(r_vals):.4f}±{np.std(r_vals):.4f}  "
          f"F1={np.mean(f_vals):.4f}±{np.std(f_vals):.4f}")

# Comparison with baseline
print("\n" + "=" * 60)
print("  COMPARISON WITH BASELINE")
print("=" * 60)
baseline_acc = 0.92  # Baseline FSDA accuracy
proposed_acc = np.mean(accuracies)
diff = proposed_acc - baseline_acc
print(f"  Baseline (FSDA)    : ~{baseline_acc:.4f}")
print(f"  Proposed (MS-CAF)  : {proposed_acc:.4f} ± {np.std(accuracies):.4f}")
print(f"  Difference         : {diff:+.4f} ({'↑ IMPROVED' if diff > 0 else '↓ Need tuning'})")

# Save summary
summary_df = pd.DataFrame([{
    'strategy': STRATEGY_KEY, 'run': r['run'], 'seed': r['seed'],
    'accuracy': r['accuracy'], 'precision': r['precision'],
    'recall': r['recall'], 'f1_score': r['f1_score'],
} for r in all_runs_results])
summary_df.to_csv(os.path.join(BASE_RESULT_DIR, 'summary.csv'), index=False)

# Zip results
zip_path = f"/kaggle/working/{STRATEGY_KEY}_complete"
shutil.make_archive(zip_path, 'zip', BASE_RESULT_DIR)
print(f"\n  ✅ Archived → {zip_path}.zip")